<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/Projects/Project-4/Spotify.data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Project 4: Spotify Music Popularity Prediction


##Problem Definition


This project is a **supervised regression** problem focused on predicting the popularity of a new song using features available prior to its release. We will employ tree-based regression models, such as **Decision Tree Regression**, **Random Forest**, and **XGBoost**, to make these predictions. Regularization techniques will be used to reduce overfitting and assist with feature selection. Model performance will be evaluated using *cross-validated* **Root Mean Squared Error** (*RMSE*), with the aim of minimizing prediction error and understanding which features contribute most to a song’s success.

##Data Collection

In [1]:
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns
import xgboost           as xgb
import pickle
import graphviz

from sklearn.model_selection import train_test_split
from sklearn                 import datasets
from sklearn.metrics         import mean_squared_error
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from IPython.display         import display
from sklearn                 import tree

In [2]:
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv'
url

'https://ddc-datascience.s3.amazonaws.com/Projects/Project.4-Spotify/Data/Spotify.csv'

In [3]:
#Look at the headers
!curl -s -I {url}

HTTP/1.1 200 OK
x-amz-id-2: 342Y8wUz/ZvF7GWkxi4iGc4cT3kkNWl/E8zrCHrwFp5utBWjATRcgMLfKkMsbcTzjfbgh05zqWiB1gEaRrH6t0HgBok60nTb
x-amz-request-id: 7K6NABVRY9WM99EG
Server: AmazonS3
Date: Tue, 14 Jul 2026 00:43:20 GMT
Last-Modified: Wed, 04 Oct 2023 17:23:56 GMT
ETag: "65b9875b11e0d7ea03ee2af024f45e99"
x-amz-server-side-encryption: AES256
Accept-Ranges: bytes
Content-Length: 738124
Content-Type: text/csv



In [4]:
#Download the file
!curl -s -O {url}

In [5]:
#Verify
!ls -la

total 6112
drwxr-xr-x 1 root root    4096 Jul 13 22:52 .
drwxr-xr-x 1 root root    4096 Jul 13 20:52 ..
drwxr-xr-x 4 root root    4096 Jun  4 13:32 .config
-rw-r--r-- 1 root root 5420664 Jul 13 21:18 rfModel.p
drwxr-xr-x 1 root root    4096 Jun  4 13:32 sample_data
-rw-r--r-- 1 root root  738124 Jul 14 00:43 Spotify.csv
-rw-r--r-- 1 root root   76773 Jul 13 22:52 spotify.parquet


In [6]:
#Look at the field names
!head -1 Spotify.csv | tr , '\n' | cat -n

     1	Index
     2	Highest Charting Position
     3	Number of Times Charted
     4	Week of Highest Charting
     5	Song Name
     6	Streams
     7	Artist
     8	Artist Followers
     9	Song ID
    10	Genre
    11	Release Date
    12	Weeks Charted
    13	Popularity
    14	Danceability
    15	Energy
    16	Loudness
    17	Speechiness
    18	Acousticness
    19	Liveness
    20	Tempo
    21	Duration (ms)
    22	Valence
    23	Chord


In [7]:
df = pd.read_csv(url)
df

,Index,Highest Charting Position,Number of Times Charted,Week of Highest Charting,Song Name,Streams,Artist,Artist Followers,Song ID,Genre,...,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,1,1,8,2021-07-23--2021-07-30,Beggin',"48,633,449",Måneskin,3377762,3Wrjm47oTz2sjIgck11l5e,"['indie rock italiano', 'italian pop']",...,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,2,2,3,2021-07-23--2021-07-30,STAY (with Justin Bieber),"47,248,719",The Kid LAROI,2230022,5HCyWlXZPP0y6Gqq8TgA20,['australian hip hop'],...,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,3,1,11,2021-06-25--2021-07-02,good 4 u,"40,162,559",Olivia Rodrigo,6266514,4ZtFanR9U6ndgddUvNcjcG,['pop'],...,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,4,3,5,2021-07-02--2021-07-09,Bad Habits,"37,799,456",Ed Sheeran,83293380,6PQ88X9TkUIAUIZJHW2upE,"['pop', 'uk pop']",...,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,5,5,1,2021-07-23--2021-07-30,INDUSTRY BABY (feat. Jack Harlow),"33,948,454",Lil Nas X,5473565,27NovPIUIRrOZoCHxABJwK,"['lgbtq+ hip hop', 'pop rap']",...,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551,1552,195,1,2019-12-27--2020-01-03,New Rules,"4,630,675",Dua Lipa,27167675,2ekn2ttSfGqwhhate0LSR0,"['dance pop', 'pop', 'uk pop']",...,0.762,0.7,-6.021,0.0694,0.00261,0.153,116.073,209320,0.608,A
1552,1553,196,1,2019-12-27--2020-01-03,Cheirosa - Ao Vivo,"4,623,030",Jorge & Mateus,15019109,2PWjKmjyTZeDpmOUa3a5da,"['sertanejo', 'sertanejo universitario']",...,0.528,0.87,-3.123,0.0851,0.24,0.333,152.37,181930,0.714,B
1553,1554,197,1,2019-12-27--2020-01-03,Havana (feat. Young Thug),"4,620,876",Camila Cabello,22698747,1rfofaqEpACxVEHIZBJe6W,"['dance pop', 'electropop', 'pop', 'post-teen ...",...,0.765,0.523,-4.333,0.03,0.184,0.132,104.988,217307,0.394,D
1554,1555,198,1,2019-12-27--2020-01-03,Surtada - Remix Brega Funk,"4,607,385","Dadá Boladão, Tati Zaqui, OIK",208630,5F8ffc8KWKNawllr5WsW0r,"['brega funk', 'funk carioca']",...,0.832,0.55,-7.026,0.0587,0.249,0.182,154.064,152784,0.881,F


##Data Cleaning

In [8]:
df.shape

(1556, 23)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1556 entries, 0 to 1555
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Index                      1556 non-null   int64 
 1   Highest Charting Position  1556 non-null   int64 
 2   Number of Times Charted    1556 non-null   int64 
 3   Week of Highest Charting   1556 non-null   object
 4   Song Name                  1556 non-null   object
 5   Streams                    1556 non-null   object
 6   Artist                     1556 non-null   object
 7   Artist Followers           1556 non-null   object
 8   Song ID                    1556 non-null   object
 9   Genre                      1556 non-null   object
 10  Release Date               1556 non-null   object
 11  Weeks Charted              1556 non-null   object
 12  Popularity                 1556 non-null   object
 13  Danceability               1556 non-null   object
 14  Energy  

###Target

In [10]:
df['Streams'] = df['Streams'].str.replace(',', '').astype(int)
df['Highest Charting Position'] = 201 - df['Highest Charting Position']
df[['Highest Charting Position', 'Streams']].describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Highest Charting Position,1556.0,1.132558e+02,5.814723e+01,1.0,64.00,121.0,164.00,200.0
Streams,1556.0,6.340219e+06,3.369479e+06,4176083.0,4915322.25,5275747.5,6455044.25,48633449.0


In [11]:
target = 'Highest Charting Position'

In [12]:
df[target].isnull().sum()

np.int64(0)

###Unique IDs

In [13]:
identifier_cols = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_cols.append(col)

print(identifier_cols)

['Index', 'Song Name', 'Streams']


In [14]:
df.drop(columns=[
  'Index',
  'Song Name',
  'Song ID',
  'Number of Times Charted',
  'Week of Highest Charting',
  'Streams',
  'Weeks Charted',
  'Popularity'
], inplace=True)
df.head()

,Highest Charting Position,Artist,Artist Followers,Genre,Release Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
0,200,Måneskin,3377762,"['indie rock italiano', 'italian pop']",2017-12-08,0.714,0.8,-4.808,0.0504,0.127,0.359,134.002,211560,0.589,B
1,199,The Kid LAROI,2230022,['australian hip hop'],2021-07-09,0.591,0.764,-5.484,0.0483,0.0383,0.103,169.928,141806,0.478,C#/Db
2,200,Olivia Rodrigo,6266514,['pop'],2021-05-21,0.563,0.664,-5.044,0.154,0.335,0.0849,166.928,178147,0.688,A
3,198,Ed Sheeran,83293380,"['pop', 'uk pop']",2021-06-25,0.808,0.897,-3.712,0.0348,0.0469,0.364,126.026,231041,0.591,B
4,196,Lil Nas X,5473565,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,0.736,0.704,-7.409,0.0615,0.0203,0.0501,149.995,212000,0.894,D#/Eb


###Rows

In [15]:
df_blank = df.apply(lambda x: x.astype(str).str.strip().eq('').sum())

df_blank[df_blank > 0]

,0
Artist Followers,11
Genre,11
Release Date,11
Danceability,11
Energy,11
Loudness,11
Speechiness,11
Acousticness,11
Liveness,11
Tempo,11


In [16]:
blank_rows = df[df.apply(lambda x: x.astype(str).str.strip().eq('').any(), axis=1)]

blank_rows

,Highest Charting Position,Artist,Artist Followers,Genre,Release Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence,Chord
35,165,The Kid LAROI,,,,,,,,,,,,,
163,196,Ariana Grande,,,,,,,,,,,,,
464,83,Rod Wave,,,,,,,,,,,,,
530,181,Ariana Grande,,,,,,,,,,,,,
636,179,Chris Rea,,,,,,,,,,,,,
654,128,Queen,,,,,,,,,,,,,
750,182,Tainy,,,,,,,,,,,,,
784,125,"Super Yei, Jone Quest",,,,,,,,,,,,,
876,37,Dalex,,,,,,,,,,,,,
1140,70,"AK AUSSERKONTROLLE, Bonez MC",,,,,,,,,,,,,


In [17]:
df[df.apply(lambda x: x.astype(str).str.strip().eq('').any(), axis=1)].index

Index([35, 163, 464, 530, 636, 654, 750, 784, 876, 1140, 1538], dtype='int64')

In [18]:
df = df.drop(index=[35, 163, 464, 530, 636, 654, 750, 784, 876, 1140, 1538])

In [19]:
#Rows with nulls
df.isnull().any(axis = 1).sum()

np.int64(0)

In [20]:
#Missing rows
df.isnull().sum().sort_values()

,0
Highest Charting Position,0
Artist,0
Artist Followers,0
Genre,0
Release Date,0
Danceability,0
Energy,0
Loudness,0
Speechiness,0
Acousticness,0


In [21]:
#Duplicate rows
df.duplicated().sum()

np.int64(0)

###Columns

In [22]:
cols = list(df.drop([target], axis=1).columns.sort_values())
cols

['Acousticness',
 'Artist',
 'Artist Followers',
 'Chord',
 'Danceability',
 'Duration (ms)',
 'Energy',
 'Genre',
 'Liveness',
 'Loudness',
 'Release Date',
 'Speechiness',
 'Tempo',
 'Valence']

####Numerical

#####Discrete

In [23]:
integer = 'Artist Followers'
df[integer] = pd.to_numeric(df[integer])
df[integer].sort_values(ascending=True)

,Artist Followers
632,4883
610,14122
149,15889
102,16074
243,17202
...,...
548,83337783
541,83337783
1349,83337783
800,83337783


In [24]:
integer = 'Duration (ms)'
df[integer] = pd.to_numeric(df[integer])
df[integer].sort_values(ascending=True)

,Duration (ms)
1499,30133
1030,30583
711,37013
1137,41867
1135,41867
...,...
42,393280
1335,457592
757,484147
265,515865


In [25]:
df_int = df.select_dtypes(include=['int64']).drop(columns=[target])
df_int.isna().sum()

,0
Artist Followers,0
Duration (ms),0


In [26]:
df_int.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Artist Followers,1545.0,1.471690e+07,1.667579e+07,4883.0,2123734.0,6852509.0,22698747.0,83337783.0
Duration (ms),1545.0,1.979408e+05,4.714893e+04,30133.0,169266.0,193591.0,218902.0,588139.0


#####Continuous

In [27]:
floats = [
  'Energy',
  'Loudness',
  'Speechiness',
  'Acousticness',
  'Liveness',
  'Tempo',
  'Danceability',
  'Valence'
]

df[floats] = df[floats].apply(pd.to_numeric, errors='coerce')
df[floats].sort_index(ascending=True)

,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Danceability,Valence
0,0.800,-4.808,0.0504,0.12700,0.3590,134.002,0.714,0.589
1,0.764,-5.484,0.0483,0.03830,0.1030,169.928,0.591,0.478
2,0.664,-5.044,0.1540,0.33500,0.0849,166.928,0.563,0.688
3,0.897,-3.712,0.0348,0.04690,0.3640,126.026,0.808,0.591
4,0.704,-7.409,0.0615,0.02030,0.0501,149.995,0.736,0.894
...,...,...,...,...,...,...,...,...
1551,0.700,-6.021,0.0694,0.00261,0.1530,116.073,0.762,0.608
1552,0.870,-3.123,0.0851,0.24000,0.3330,152.370,0.528,0.714
1553,0.523,-4.333,0.0300,0.18400,0.1320,104.988,0.765,0.394
1554,0.550,-7.026,0.0587,0.24900,0.1820,154.064,0.832,0.881


In [28]:
df_flt = df.select_dtypes(include=['float64'])
df_flt.isna().sum()

,0
Danceability,0
Energy,0
Loudness,0
Speechiness,0
Acousticness,0
Liveness,0
Tempo,0
Valence,0


In [29]:
df_flt.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Danceability,1545.0,0.689997,0.142444,0.150000,0.5990,0.7070,0.796,0.980
Energy,1545.0,0.633495,0.161577,0.054000,0.5320,0.6420,0.752,0.970
Loudness,1545.0,-6.348474,2.509281,-25.166000,-7.4910,-5.9900,-4.711,1.509
Speechiness,1545.0,0.123656,0.110383,0.023200,0.0456,0.0765,0.165,0.884
Acousticness,1545.0,0.248695,0.250326,0.000025,0.0485,0.1610,0.388,0.994
Liveness,1545.0,0.181202,0.144071,0.019700,0.0966,0.1240,0.217,0.962
Tempo,1545.0,122.811023,29.591088,46.718000,97.9600,122.0120,143.860,205.272
Valence,1545.0,0.514704,0.227326,0.032000,0.3430,0.5120,0.691,0.979


####Categorical

In [30]:
df_obj = df.select_dtypes(include=['object'])
df_obj

,Artist,Genre,Release Date,Chord
0,Måneskin,"['indie rock italiano', 'italian pop']",2017-12-08,B
1,The Kid LAROI,['australian hip hop'],2021-07-09,C#/Db
2,Olivia Rodrigo,['pop'],2021-05-21,A
3,Ed Sheeran,"['pop', 'uk pop']",2021-06-25,B
4,Lil Nas X,"['lgbtq+ hip hop', 'pop rap']",2021-07-23,D#/Eb
...,...,...,...,...
1551,Dua Lipa,"['dance pop', 'pop', 'uk pop']",2017-06-02,A
1552,Jorge & Mateus,"['sertanejo', 'sertanejo universitario']",2019-10-11,B
1553,Camila Cabello,"['dance pop', 'electropop', 'pop', 'post-teen ...",2018-01-12,D
1554,"Dadá Boladão, Tati Zaqui, OIK","['brega funk', 'funk carioca']",2019-09-25,F


In [31]:
df.drop(columns=[
  'Artist', #Artist Followers can be used to represent the artist.
  'Genre',
  'Release Date',
  'Chord'
], inplace=True)

###Copy

In [32]:
df_clean = df.copy()
df_clean

,Highest Charting Position,Artist Followers,Danceability,Energy,Loudness,Speechiness,Acousticness,Liveness,Tempo,Duration (ms),Valence
0,200,3377762,0.714,0.800,-4.808,0.0504,0.12700,0.3590,134.002,211560,0.589
1,199,2230022,0.591,0.764,-5.484,0.0483,0.03830,0.1030,169.928,141806,0.478
2,200,6266514,0.563,0.664,-5.044,0.1540,0.33500,0.0849,166.928,178147,0.688
3,198,83293380,0.808,0.897,-3.712,0.0348,0.04690,0.3640,126.026,231041,0.591
4,196,5473565,0.736,0.704,-7.409,0.0615,0.02030,0.0501,149.995,212000,0.894
...,...,...,...,...,...,...,...,...,...,...,...
1551,6,27167675,0.762,0.700,-6.021,0.0694,0.00261,0.1530,116.073,209320,0.608
1552,5,15019109,0.528,0.870,-3.123,0.0851,0.24000,0.3330,152.370,181930,0.714
1553,4,22698747,0.765,0.523,-4.333,0.0300,0.18400,0.1320,104.988,217307,0.394
1554,3,208630,0.832,0.550,-7.026,0.0587,0.24900,0.1820,154.064,152784,0.881


##Save as a Parquet

In [38]:
from google.colab import userdata
import os

In [33]:
parquet_file = 'spotify.parquet'
parquet_file

'spotify.parquet'

In [34]:
df_clean.to_parquet(parquet_file, index=False)

In [35]:
!ls -l --si {parquet_file}

-rw-r--r-- 1 root root 77k Jul 14 00:43 spotify.parquet


In [36]:
df2 = pd.read_parquet(parquet_file)
df2.shape

(1545, 11)

In [39]:
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

'hf_rP ... BIc'

In [40]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account

'stephanie465337'

In [41]:
hf_repo = "Data_Science-21"
os.environ["HF_REPO"] = hf_repo
hf_repo

'Data_Science-21'

In [42]:
!hf auth login --token $HF_TOKEN

Hint: A new version of huggingface_hub (1.23.0) is available! You are using version 1.20.1.
To update, run: hf update
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [45]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  spotify.parquet

In [46]:
print(hf_upload.stdout)

✓ Uploaded
  url: https://huggingface.co/datasets/Stephanie465337/Data_Science-21/commit/fae71bebbc708a941125a4eaf0c4ebf790cd33bb



In [47]:
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/spotify.parquet"
hf_url

'https://huggingface.co/datasets/stephanie465337/Data_Science-21/resolve/main/spotify.parquet'

In [48]:
df3 = pd.read_parquet(hf_url)
df3.shape

(1545, 11)

In [49]:
df3.iloc[:,:5].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1545 entries, 0 to 1544
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Highest Charting Position  1545 non-null   int64  
 1   Artist Followers           1545 non-null   int64  
 2   Danceability               1545 non-null   float64
 3   Energy                     1545 non-null   float64
 4   Loudness                   1545 non-null   float64
dtypes: float64(3), int64(2)
memory usage: 60.5 KB
